In [14]:
# Cell 0 — Setup
import sys
from pathlib import Path

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name.lower() == "sandbox" else CWD
if not (ROOT / "raw_data").exists() and (ROOT.parent / "raw_data").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW = ROOT / "raw_data"
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("RAW exists:", RAW.exists(), "| DATA:", DATA)
for f in ["BTC.csv","Gas Natural Futures.csv","IDY.csv","macro_data.csv","SPY.csv","XAU-USD.csv"]:
    p = RAW / f
    print(f"{f:25s} -> {p.exists()}")


ROOT: c:\Users\kraus\projetos - git\quant
RAW exists: True | DATA: c:\Users\kraus\projetos - git\quant\data
BTC.csv                   -> True
Gas Natural Futures.csv   -> True
IDY.csv                   -> True
macro_data.csv            -> True
SPY.csv                   -> True
XAU-USD.csv               -> True


In [17]:
# Cell 1 — Helpers robustos (autodetect sep/encoding) + loaders

import pandas as pd
import numpy as np
from typing import Sequence

PRICE_CANDIDATES = [
    # pt-BR / Investing
    "Último","Ultimo","Fechamento","Fechamento Ajustado","Abertura","Preço","Preco",
    # en
    "Close","Adj Close","Close*","Close/Last","Price","Last","close","adj_close",
]

DATE_FORMATS: Sequence[str] = ("%Y-%m-%d","%d/%m/%Y","%m/%d/%Y","%d-%m-%Y","%Y/%m/%d")

def _read_csv_smart(path):
    """Tenta detectar separador e encoding (utf-8-sig / latin-1)."""
    try:
        return pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, sep=None, engine="python", encoding="latin-1")

def _coerce_datetime(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        med = pd.to_numeric(s, errors="coerce").dropna().astype(float).median()
        unit = "ms" if med and med > 10_000_000_000 else "s"
        return pd.to_datetime(s, unit=unit, errors="coerce")
    for fmt in DATE_FORMATS:
        dt = pd.to_datetime(s, format=fmt, errors="coerce")
        if dt.notna().mean() > 0.95:
            return dt
    return pd.to_datetime(s, errors="coerce")

def _coerce_numeric(s: pd.Series) -> pd.Series:
    if s.dtype != object:
        return pd.to_numeric(s, errors="coerce")
    return (s.astype(str)
              .str.replace("\u00a0","", regex=False)  # NBSP
              .str.replace("’","", regex=False)
              .str.replace(".","", regex=False)       # milhar
              .str.replace(",",".", regex=False))     # decimal
    # depois:
    # return pd.to_numeric(...)

def read_price_series(path, out_name, price_col=None):
    df = _read_csv_smart(path)
    date_col = df.columns[0]
    df[date_col] = _coerce_datetime(df[date_col])
    df = df.dropna(subset=[date_col]).sort_values(date_col)

    # 1) coluna explícita
    if price_col and price_col in df.columns:
        chosen = price_col
    else:
        # 2) tenta por nome (casefold p/ acentos)
        lower_map = {c.casefold(): c for c in df.columns}
        chosen = None
        for cand in PRICE_CANDIDATES:
            key = cand.casefold()
            if key in lower_map:
                chosen = lower_map[key]
                break
        # 3) fallback: coluna não-data com mais observações numéricas
        if chosen is None:
            candidates = [c for c in df.columns if c != date_col]
            counts = {}
            for c in candidates:
                counts[c] = _coerce_numeric(df[c]).notna().sum()
            if not counts:
                raise ValueError(f"Nenhuma coluna candidata em {path}")
            chosen, count = max(counts.items(), key=lambda kv: kv[1])
            if count < 0.6 * len(df):
                raise ValueError(f"Sem coluna de preço confiável em {path}. Contagens: {counts}")

    s = df[[date_col, chosen]].copy()
    s[chosen] = _coerce_numeric(s[chosen]).pipe(pd.to_numeric, errors="coerce")
    s = s.dropna(subset=[chosen])
    s = s.set_index(date_col)[chosen].astype(float)
    s.index = pd.to_datetime(s.index)
    s = s[~s.index.duplicated(keep="last")].sort_index()
    s.name = out_name
    print(f"[read_price_series] {path.name} -> preço '{chosen}' | n={len(s)}")
    return s

def load_macro(path):
    m = _read_csv_smart(path)
    dcol = "Date" if "Date" in m.columns else m.columns[0]
    m[dcol] = _coerce_datetime(m[dcol])
    m = m.dropna(subset=[dcol]).set_index(dcol).sort_index()
    for c in m.columns:
        m[c] = _coerce_numeric(m[c]).pipe(pd.to_numeric, errors="coerce")
    m = m.dropna(how="all")
    m.columns = [c.strip().lower() for c in m.columns]
    print("[load_macro] colunas:", list(m.columns))
    return m


In [18]:
# Cell 2 — Carrega BTC, XAU, NG e monta prices
BTC = read_price_series(RAW / "BTC.csv", "BTC")
XAU = read_price_series(RAW / "XAU-USD.csv", "XAU")
NG  = read_price_series(RAW / "Gas Natural Futures.csv", "NG")

prices = pd.concat([BTC, XAU, NG], axis=1, join="inner").sort_index()
print("prices shape:", prices.shape, "|", prices.index.min().date(), "→", prices.index.max().date())
prices.head()


[read_price_series] BTC.csv -> preço 'Último' | n=1969
[read_price_series] XAU-USD.csv -> preço 'Último' | n=11888
[read_price_series] Gas Natural Futures.csv -> preço 'Último' | n=6877
prices shape: (1383, 3) | 2010-01-08 → 2025-10-10


C:\Users\kraus\AppData\Local\Temp\ipykernel_50052\521826358.py:32: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors="coerce")
C:\Users\kraus\AppData\Local\Temp\ipykernel_50052\521826358.py:32: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors="coerce")


,BTC,XAU,NG
Data,,,
2010-01-08,0.1,1136.60,5.749
2010-01-11,0.2,1153.00,5.454
2010-01-12,0.2,1127.70,5.591
2010-02-08,0.1,1063.75,5.401
2010-02-09,0.1,1077.30,5.290


In [20]:
# Cell 3 — Persistência de preços e retornos
CLEAN_PRICES = DATA / "clean_prices.csv"
prices.to_csv(CLEAN_PRICES)
print("salvo:", CLEAN_PRICES)

logrets = np.log(prices).diff().dropna()
logrets.columns = [c + "_ret" for c in logrets.columns]

LOG_RETURNS = DATA / "log_returns.csv"
logrets.to_csv(LOG_RETURNS)
print("salvo:", LOG_RETURNS)

logrets.describe().T[["mean","std","min","max"]]


salvo: c:\Users\kraus\projetos - git\quant\data\clean_prices.csv
salvo: c:\Users\kraus\projetos - git\quant\data\log_returns.csv


,mean,std,min,max
BTC_ret,0.010084,0.648459,-4.525675,3.268531
XAU_ret,0.000914,0.015555,-0.102372,0.132761
NG_ret,-0.000446,0.053408,-0.821684,0.325180


In [21]:
# Cell 4 — EWMA vol e retorno
def ewma_vol(series, span=20):
    return series.ewm(span=span, adjust=False).std() * np.sqrt(252)

def ewma_ret(series, span=20):
    return series.ewm(span=span, adjust=False).mean()

ew_cols = {}
for c in logrets.columns:
    ew_cols[f"{c}_ewma_vol_20"] = ewma_vol(logrets[c], 20)
    ew_cols[f"{c}_ewma_vol_60"] = ewma_vol(logrets[c], 60)
    ew_cols[f"{c}_ewma_ret_20"] = ewma_ret(logrets[c], 20)
    ew_cols[f"{c}_ewma_ret_60"] = ewma_ret(logrets[c], 60)

ewma_df = pd.concat(ew_cols, axis=1)
EWMA_CSV = DATA / "ewma_metrics.csv"
ewma_df.to_csv(EWMA_CSV)
print("salvo:", EWMA_CSV, "| shape:", ewma_df.shape)
ewma_df.head()


salvo: c:\Users\kraus\projetos - git\quant\data\ewma_metrics.csv | shape: (1382, 12)


,BTC_ret_ewma_vol_20,BTC_ret_ewma_vol_60,BTC_ret_ewma_ret_20,BTC_ret_ewma_ret_60,XAU_ret_ewma_vol_20,XAU_ret_ewma_vol_60,XAU_ret_ewma_ret_20,XAU_ret_ewma_ret_60,NG_ret_ewma_vol_20,NG_ret_ewma_vol_60,NG_ret_ewma_ret_20,NG_ret_ewma_ret_60
Data,,,,,,,,,,,,
2010-01-11,NaN,NaN,0.693147,0.693147,NaN,NaN,0.014326,0.014326,NaN,NaN,-0.052677,-0.052677
2010-01-12,7.780558,7.780558,0.627133,0.670421,0.409857,0.409857,0.010848,0.013129,0.869773,0.869773,-0.045297,-0.050136
2010-02-08,12.283721,12.300092,0.501392,0.625714,0.644687,0.645623,0.004255,0.010784,0.619945,0.627541,-0.044276,-0.049626
2010-02-09,10.606057,10.877484,0.453641,0.605199,0.527600,0.527054,0.005056,0.010846,0.531116,0.547388,-0.042037,-0.048680
2010-02-10,9.588088,10.066796,0.410437,0.585356,0.458119,0.464613,0.004180,0.010354,0.532689,0.554965,-0.037997,-0.047071


In [22]:
# Cell 5 — Macro join
macro = load_macro(RAW / "macro_data.csv")
keep = [c for c in ["vix","juros_10a","dolar","petroleo"] if c in macro.columns]
if not keep:
    keep = [c for c in macro.columns if pd.api.types.is_numeric_dtype(macro[c])]
print("macro cols usadas:", keep)

panel = logrets.join(macro[keep], how="inner").dropna()
PANEL_CSV = DATA / "panel_debug.csv"
panel.to_csv(PANEL_CSV)
print("panel:", panel.shape, "|", panel.index.min().date(), "→", panel.index.max().date())
print("salvo:", PANEL_CSV)
panel.head()


[load_macro] colunas: ['vix', 'juros_10a', 'dolar', 'petroleo']
macro cols usadas: ['vix', 'juros_10a', 'dolar', 'petroleo']
panel: (1352, 7) | 2010-01-11 → 2025-10-10
salvo: c:\Users\kraus\projetos - git\quant\data\panel_debug.csv


,BTC_ret,XAU_ret,NG_ret,vix,juros_10a,dolar,petroleo
2010-01-11,0.693147,0.014326,-0.052677,82.519997,77.000000,3.818,17.549999
2010-01-12,0.000000,-0.022187,0.024809,80.790001,76.949997,3.719,18.250000
2010-02-08,-0.693147,-0.058380,-0.034574,71.889999,80.300003,3.592,26.510000
2010-02-09,0.000000,0.012658,-0.020766,73.750000,79.860001,3.633,26.000000
2010-02-10,0.000000,-0.004139,0.000378,74.519997,80.029999,3.694,25.400000


In [23]:
# Cell 7 — Johansen nos níveis de preço (BTC, XAU, NG)
from statsmodels.tsa.vector_ar.vecm import coint_johansen

prices_j = prices.dropna()
res = coint_johansen(prices_j, det_order=0, k_ar_diff=1)

import numpy as np
out = []
out.append("=== Johansen: [BTC, XAU, NG] ===")
out.append(f"datas: {prices_j.index.min().date()} → {prices_j.index.max().date()}  n={len(prices_j)}")
out.append("eig (autovalores): " + np.array2string(res.eig, precision=5))
out.append("trace stats (lr1): " + np.array2string(res.lr1, precision=5))
out.append("crit (90/95/99) por rank (linhas):")
out.append(str(res.cvt))

JOH_TXT = DATA / "johansen_btc_xau_ng.txt"
with open(JOH_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(out))

print("salvo:", JOH_TXT)
print("\n".join(out[:6]))


salvo: c:\Users\kraus\projetos - git\quant\data\johansen_btc_xau_ng.txt
=== Johansen: [BTC, XAU, NG] ===
datas: 2010-01-08 → 2025-10-10  n=1383
eig (autovalores): [0.05425 0.01912 0.00651]
trace stats (lr1): [112.71627  35.68624   9.02194]
crit (90/95/99) por rank (linhas):
[[27.0669 29.7961 35.4628]
 [13.4294 15.4943 19.9349]
 [ 2.7055  3.8415  6.6349]]
